In [4]:
import cv2
import numpy as np
import glob
import os

print("--- 切り抜き確認 ＆ メッシュ解析ツール (中央フォーカス版) ---")
print("※ check_crop フォルダに切り抜き結果を保存します")
print("-" * 75)

image_files = list(set(glob.glob("*.jpg") + glob.glob("*.JPG")))
image_files.sort()

os.makedirs("check_crop", exist_ok=True)

for img_path in image_files:
    nparr = np.fromfile(img_path, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    if img is None:
        continue

    # --- ★改良1：画像の「端っこ（暗箱の影）」を完全に無視する ---
    height, width = img.shape[:2]
    # 画像の上下左右から20%ずつ内側の「中央エリア」だけを切り出す
    cx1, cx2 = int(width * 0.2), int(width * 0.8)
    cy1, cy2 = int(height * 0.2), int(height * 0.8)
    
    # これで黒い箱の壁が除外された、紫の紙と膜だけの画像になります
    center_img = img[cy1:cy2, cx1:cx2]

    # この「中央エリア」の中で膜を探す
    gray = cv2.cvtColor(center_img, cv2.COLOR_BGR2GRAY)
    
    # ★改良2：紫の紙は明るいので、閾値を少し高め(120)に設定して膜をくっきり分離
    _, thresh = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        print(f"{os.path.basename(img_path):30} | 中央に膜が見つかりませんでした")
        continue

    # 一番面積が大きい輪郭（＝ペロブスカイト膜）を見つける
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    
    # 膜が小さいので、切り抜きの余白(マージン)を少し小さくします
    margin = 5
    y1, y2 = max(0, y+margin), min(center_img.shape[0], y+h-margin)
    x1, x2 = max(0, x+margin), min(center_img.shape[1], x+w-margin)
    
    # 膜の切り抜き完了！
    cropped_img = center_img[y1:y2, x1:x2]

    # --- 切り抜いた画像を check_crop フォルダに保存 ---
    filename = os.path.basename(img_path)
    save_path = os.path.join("check_crop", "crop_" + filename)
    
    _, encoded_img = cv2.imencode(".jpg", cropped_img)
    with open(save_path, "wb") as f:
        encoded_img.tofile(f)

    # --- メッシュ解析処理 ---
    mesh_size = 100
    try:
        mesh_img = cv2.resize(cropped_img, (mesh_size, mesh_size), interpolation=cv2.INTER_AREA)
    except cv2.error:
        continue

    rgb_img = cv2.cvtColor(mesh_img, cv2.COLOR_BGR2RGB)
    r, g, b = cv2.split(rgb_img)

    r_std = np.std(r)
    g_std = np.std(g)
    b_std = np.std(b)

    # 閾値を80にしていますが、後で膜の色味に合わせて調整可能です
    good_pixels = np.sum((r < 80) & (g < 80) & (b < 80))
    total_pixels = mesh_size * mesh_size
    good_ratio = (good_pixels / total_pixels) * 100

    print(f"{filename:30} | {good_ratio:5.1f}% | R:{r_std:4.1f} | G:{g_std:4.1f} | B:{b_std:4.1f}")

print("-" * 75)

--- 切り抜き確認 ＆ メッシュ解析ツール (中央フォーカス版) ---
※ check_crop フォルダに切り抜き結果を保存します
---------------------------------------------------------------------------
7_2_FACsPbI3_RbCl_100C_0%.jpg  |   1.5% | R:26.5 | G:21.2 | B:32.7
7_2_FACsPbI3_RbCl_100C_10%.jpg |   0.0% | R:11.5 | G: 8.0 | B:12.9
7_2_FACsPbI3_RbCl_100C_5%.jpg  |   0.0% | R:14.7 | G: 9.2 | B:14.3
7_2_FACsPbI3_RbCl_130C_0%.jpg  |   0.0% | R:32.2 | G:21.8 | B:31.5
7_2_FACsPbI3_RbCl_130C_10%.jpg |   0.0% | R:11.1 | G: 7.8 | B:11.9
7_2_FACsPbI3_RbCl_130C_5%.jpg  |   0.0% | R:13.9 | G:11.1 | B:16.3
---------------------------------------------------------------------------


In [5]:
import cv2
import numpy as np
import glob
import os

# --- 追加関数：4つの角の座標を正しい順番（左上、右上、右下、左下）に並び替える ---
def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]      # 左上
    rect[2] = pts[np.argmax(s)]      # 右下
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]   # 右上
    rect[3] = pts[np.argmax(diff)]   # 左下
    return rect

print("--- 傾き補正 ＆ メッシュ解析ツール (射影変換・完全版) ---")
print("※ check_crop フォルダに切り抜き結果を保存します")
print("-" * 75)

image_files = list(set(glob.glob("*.jpg") + glob.glob("*.JPG")))
image_files.sort()

os.makedirs("check_crop", exist_ok=True)

for img_path in image_files:
    nparr = np.fromfile(img_path, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    
    if img is None:
        continue

    # --- 画像の「端っこ（暗箱の影）」を無視する（中央フォーカス） ---
    height, width = img.shape[:2]
    cx1, cx2 = int(width * 0.2), int(width * 0.8)
    cy1, cy2 = int(height * 0.2), int(height * 0.8)
    center_img = img[cy1:cy2, cx1:cx2]

    # 膜を探す
    gray = cv2.cvtColor(center_img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        print(f"{os.path.basename(img_path):30} | 中央に膜が見つかりませんでした")
        continue

    # 一番面積が大きい輪郭（＝ペロブスカイト膜）を見つける
    largest_contour = max(contours, key=cv2.contourArea)
    
    # --- ★ここからが「傾き補正（射影変換）」の処理 ---
    # 1. 傾きを考慮した「最小の四角形」を取得し、4つの角の座標を取り出す
    rect = cv2.minAreaRect(largest_contour)
    box = cv2.boxPoints(rect)
    box = np.float32(box)

    # 2. 4つの角を左上・右上・右下・左下の順に整理する
    rect_pts = order_points(box)

    # 3. 変換後の画像サイズを決定（ペロブスカイト基板は正方形なので、縦横を同じサイズに強制します）
    side_length = int(max(rect[1][0], rect[1][1]))
    
    # 4. 変換先の真っ直ぐな正方形の4つの角の座標
    dst_pts = np.float32([
        [0, 0],
        [side_length - 1, 0],
        [side_length - 1, side_length - 1],
        [0, side_length - 1]
    ])

    # 5. 変換マトリックスを計算し、画像を真っ直ぐに引き伸ばす
    matrix = cv2.getPerspectiveTransform(rect_pts, dst_pts)
    warped_img = cv2.warpPerspective(center_img, matrix, (side_length, side_length))

    # 6. 端のノイズや背景色を消すために、補正後の画像から少し内側(マージン)を切り抜く
    margin = 5
    if side_length > margin * 2:
        cropped_img = warped_img[margin:side_length-margin, margin:side_length-margin]
    else:
        cropped_img = warped_img

    # --- 切り抜いた画像を check_crop フォルダに保存 ---
    filename = os.path.basename(img_path)
    save_path = os.path.join("check_crop", "crop_" + filename)
    
    _, encoded_img = cv2.imencode(".jpg", cropped_img)
    with open(save_path, "wb") as f:
        encoded_img.tofile(f)

    # --- メッシュ解析処理（真っ直ぐになった画像に対して実行） ---
    mesh_size = 100
    try:
        mesh_img = cv2.resize(cropped_img, (mesh_size, mesh_size), interpolation=cv2.INTER_AREA)
    except cv2.error:
        continue

    rgb_img = cv2.cvtColor(mesh_img, cv2.COLOR_BGR2RGB)
    r, g, b = cv2.split(rgb_img)

    r_std = np.std(r)
    g_std = np.std(g)
    b_std = np.std(b)

    good_pixels = np.sum((r < 80) & (g < 80) & (b < 80))
    total_pixels = mesh_size * mesh_size
    good_ratio = (good_pixels / total_pixels) * 100

    print(f"{filename:30} | {good_ratio:5.1f}% | R:{r_std:4.1f} | G:{g_std:4.1f} | B:{b_std:4.1f}")

print("-" * 75)

--- 傾き補正 ＆ メッシュ解析ツール (射影変換・完全版) ---
※ check_crop フォルダに切り抜き結果を保存します
---------------------------------------------------------------------------
7_2_FACsPbI3_RbCl_100C_0%.jpg  |   1.5% | R:16.9 | G:12.7 | B:21.4
7_2_FACsPbI3_RbCl_100C_10%.jpg |   0.0% | R:11.4 | G: 8.0 | B:12.8
7_2_FACsPbI3_RbCl_100C_5%.jpg  |   0.0% | R:14.7 | G: 9.3 | B:14.4
7_2_FACsPbI3_RbCl_130C_0%.jpg  |   0.0% | R:29.5 | G:18.9 | B:27.4
7_2_FACsPbI3_RbCl_130C_10%.jpg |   0.0% | R:11.6 | G: 8.5 | B:12.7
7_2_FACsPbI3_RbCl_130C_5%.jpg  |   0.0% | R:10.6 | G: 7.3 | B:11.4
---------------------------------------------------------------------------
